# ImageNet100 用の DeepInversion+mmd ノートブック

In [1]:
import os
import sys
import numpy as np
import json
import random
import collections


import torch
import torch.optim as optim
import torchvision.utils as vutils

import torch.nn as nn
import torch.nn.functional as F



In [2]:
# 使用するgpuを指定
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

## パスの設定

In [3]:
# ベース部分のパス
ckpt_path = "/home/kouyou/ContinualLearning/repexp/NeurIPS2024-PRL/checkpoint"


# imagenet100のbaseline用パス
base_cifar100_path = "baseline/imagenet100"

# baseline
method = "baseline_mu"
baseline_path = os.path.join(ckpt_path, method, base_cifar100_path, "50/5/10_10_0_0_1.0")

## 色々と設定

In [4]:

# プロジェクト root を sys.path に追加
project_root = "/home/kouyou/ContinualLearning/repexp/NeurIPS2024-PRL"
sys.path.append(project_root)


from utils import factory
import models


# --- 1) 設定を記述した jsonファイル の内容を読む ---
with open(os.path.join(project_root, "exps", "BASELINE", "imnet100.json")) as f:
    args = json.load(f)

args["device"] = ["0"]            # 必要に応じて
# args["model_name"] = "baseline" 


# --- 2) learner とネットワークの作成 ---
learner = factory.get_model(args["model_name"], args)
net = learner._network


# --- 3) checkpoint の読み込み ---
ckpt_dir = baseline_path
ckpt_file = os.path.join(ckpt_dir, "phase0.pkl")       # 読み込むモデルの指定

ckpt = torch.load(ckpt_file, map_location="cuda:0")
state_dict = ckpt["model_state_dict"]
print(state_dict.keys())

# fc の出力次元を checkpoint から取得
num_outputs = state_dict["fc.weight"].shape[0]
print("num_outputs: ", num_outputs)

# fc層の出力次元数を変更
net.update_fc(num_outputs)

# state_dict の読み込み
net.load_state_dict(state_dict)

# protos, forget_classes も保存されていれば復元
if "protos" in ckpt:
    net._protos = ckpt["protos"]
    # assert False
if "forget_classes" in ckpt and hasattr(net, "forget_classes"):
    net.forget_classes = ckpt["forget_classes"]

net.cuda().eval()

# 忘却クラスや class_order を取り出す
forget_classes = ckpt.get("forget_classes", None)
class_order = ckpt.get("_class_order", None)
print(class_order)

<ipython-input-4-e66ad6e43c64>:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_file, map_location="cuda:0")


odict_keys(['convnet.conv1.0.weight', 'convnet.conv1.1.weight', 'convnet.conv1.1.bias', 'convnet.conv1.1.running_mean', 'convnet.conv1.1.running_var', 'convnet.conv1.1.num_batches_tracked', 'convnet.layer1.0.conv1.weight', 'convnet.layer1.0.bn1.weight', 'convnet.layer1.0.bn1.bias', 'convnet.layer1.0.bn1.running_mean', 'convnet.layer1.0.bn1.running_var', 'convnet.layer1.0.bn1.num_batches_tracked', 'convnet.layer1.0.conv2.weight', 'convnet.layer1.0.bn2.weight', 'convnet.layer1.0.bn2.bias', 'convnet.layer1.0.bn2.running_mean', 'convnet.layer1.0.bn2.running_var', 'convnet.layer1.0.bn2.num_batches_tracked', 'convnet.layer1.1.conv1.weight', 'convnet.layer1.1.bn1.weight', 'convnet.layer1.1.bn1.bias', 'convnet.layer1.1.bn1.running_mean', 'convnet.layer1.1.bn1.running_var', 'convnet.layer1.1.bn1.num_batches_tracked', 'convnet.layer1.1.conv2.weight', 'convnet.layer1.1.bn2.weight', 'convnet.layer1.1.bn2.bias', 'convnet.layer1.1.bn2.running_mean', 'convnet.layer1.1.bn2.running_var', 'convnet.layer

## Hookの設定

In [5]:
class DeepInversionFeatureHook():
    '''
    Implementation of the forward hook to track feature statistics and compute a loss on them.
    Will compute mean and variance, and will use l2 as a loss
    '''

    def __init__(self, module):
        self.hook = module.register_forward_hook(self.hook_fn)


    def hook_fn(self, module, input, output):
        # hook co compute deepinversion's feature distribution regularization
        nch = input[0].shape[1]

        mean = input[0].mean([0, 2, 3])
        var = input[0].permute(1, 0, 2, 3).contiguous().view([nch, -1]).var(1, unbiased=False)

        # forcing mean and variance to match between two distributions
        # other ways might work better, e.g. KL divergence
        r_feature = torch.norm(module.running_var.data.type(var.type()) - var, 2) + torch.norm(
            module.running_mean.data.type(var.type()) - mean, 2)

        self.r_feature = r_feature
        # must have no output

    def close(self):
        self.hook.remove()


In [6]:
def lr_policy(lr_fn):
    def _alr(optimizer, iteration, epoch):
        lr = lr_fn(iteration, epoch)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

    return _alr


def lr_cosine_policy(base_lr, warmup_length, epochs):
    def _lr_fn(iteration, epoch):
        if epoch < warmup_length:
            lr = base_lr * (epoch + 1) / warmup_length
        else:
            e = epoch - warmup_length
            es = epochs - warmup_length
            lr = 0.5 * (1 + np.cos(np.pi * e / es)) * base_lr
        return lr

    return lr_policy(_lr_fn)


def clip(image_tensor, use_fp16=False):
    '''
    adjust the input based on mean and variance
    '''
    if use_fp16:
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float16)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float16)
    else:
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
    for c in range(3):
        m, s = mean[c], std[c]
        image_tensor[:, c] = torch.clamp(image_tensor[:, c], -m / s, (1 - m) / s)
    return image_tensor

## 最適化対象の準備など

## 交差エントロピー損失+プロトタイプ距離損失+MMD

In [9]:
# ============================================
# RBF カーネル & MMD 損失の定義
# ============================================
import torch
import torch.nn as nn


class RBF(nn.Module):
    """
    RBF (Gaussian) カーネル行列を計算するクラス。
    
    k(x_i, x_j) = exp(- ||x_i - x_j||^2 / bandwidth)
    
    - bandwidth を指定しない場合は，
      バッチ内の平均距離^2 から簡易的に推定します。
    """

    def __init__(self, bandwidth: float = None):
        """
        Args:
            bandwidth (float or None):
                カーネルの帯域幅。
                None のときはデータから自動推定。
        """
        super().__init__()
        self.bandwidth = bandwidth

    def _get_bandwidth(self, dist2: torch.Tensor) -> torch.Tensor:
        """
        L2 距離^2 の行列から bandwidth を決めるヘルパー関数。
        dist2: 形状 (N, N)
        """
        if self.bandwidth is not None:
            # ユーザ指定がある場合はそれを使う
            return torch.tensor(self.bandwidth, device=dist2.device, dtype=dist2.dtype)

        n = dist2.shape[0]
        if n <= 1:
            # サンプルが1つ以下だと平均距離が定義できないので適当に1.0
            return torch.tensor(1.0, device=dist2.device, dtype=dist2.dtype)

        # 全要素（対角も含む）の平均距離^2 を bandwidth として使う簡易ヒューリスティック
        # （より厳密にやるなら対角を除いたり，median heuristic にしてもよい）
        bw = dist2.sum() / (n * n)
        return bw.detach()

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        """
        入力:
            X: 形状 (N, D) の特徴ベクトル集合
        出力:
            K: 形状 (N, N) の RBF カーネル行列
        """
        # 全ペアの L2 距離^2 を計算
        # torch.cdist(X, X) は (N, N) の距離行列を返す
        dist2 = torch.cdist(X, X) ** 2  # (N, N)

        bw = self._get_bandwidth(dist2)  # スカラー（tensor）
        # k(x_i, x_j) = exp(- dist2 / bw)
        K = torch.exp(- dist2 / (bw + 1e-8))
        return K


class MMDLoss(nn.Module):
    """
    RBF カーネルを用いた MMD^2 損失。
    
    MMD^2(X, Y) = E[k(X, X)] + E[k(Y, Y)] - 2 E[k(X, Y)]
    
    ここでは biased な推定量（全要素の単純平均）を用いています。
    """

    def __init__(self, kernel: nn.Module = None):
        """
        Args:
            kernel: カーネルとして使うモジュール。
                    デフォルトは上で定義した RBF。
        """
        super().__init__()
        self.kernel = kernel if kernel is not None else RBF()

    def forward(self, X: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
        """
        Args:
            X: 形状 (N_x, D) のテンソル
            Y: 形状 (N_y, D) のテンソル
        Returns:
            mmd2: スカラーの MMD^2
        """
        # X と Y を縦に連結して，まとめてカーネル行列を計算
        Z = torch.vstack([X, Y])          # (N_x + N_y, D)
        K = self.kernel(Z)                # (N_x + N_y, N_x + N_y)

        n_x = X.shape[0]

        # ブロックを切り出す
        K_xx = K[:n_x, :n_x]              # X 対 X
        K_xy = K[:n_x, n_x:]              # X 対 Y
        K_yy = K[n_x:, n_x:]              # Y 対 Y

        # 全要素の平均で expectation を近似
        XX = K_xx.mean()
        XY = K_xy.mean()
        YY = K_yy.mean()

        # MMD^2 = E[k(X,X)] - 2E[k(X,Y)] + E[k(Y,Y)]
        mmd2 = XX - 2.0 * XY + YY
        return mmd2


# --------------------------------------------------
# 3. MMD のバッチ内計算ヘルパー
# --------------------------------------------------
def compute_batch_mmd(feat: torch.Tensor,
                      targets: torch.Tensor,
                      real_features: dict,
                      max_real_per_class: int = 100) -> torch.Tensor:
    """
    各クラス c について：
      MMD^2( DI特徴_c , 実特徴_c ) を計算し，クラス平均を返す。
    feat:    (bs, D)   DI 画像の特徴
    targets: (bs,)     クラスID
    """
    uniq_classes = targets.unique().tolist()
    mmd_list = []

    for c in uniq_classes:
        c_int = int(c)
        # DI 側
        mask_di = (targets == c)
        feat_di_c = feat[mask_di]          # (n_c, D)
        if feat_di_c.shape[0] == 0:
            continue

        # 実側
        if c_int not in real_features:
            continue
        feat_real_c = real_features[c_int]  # (N_real, D) on CPU
        if feat_real_c.shape[0] == 0:
            continue

        # 実側を max_real_per_class 個に制限（計算コスト対策）
        if feat_real_c.shape[0] > max_real_per_class:
            idx = torch.randperm(feat_real_c.shape[0])[:max_real_per_class]
            feat_real_c = feat_real_c[idx]

        feat_real_c = feat_real_c.to(feat.device)

        # MMD^2 を計算
        mmd2_c = mmd_loss_fn(feat_di_c, feat_real_c)
        mmd_list.append(mmd2_c)

    if len(mmd_list) == 0:
        return torch.tensor(0.0, device=feat.device)
    else:
        return torch.stack(mmd_list).mean()

In [8]:
iterations = 4000
start_noise = True
# args.detach_student = False

resolution = 224
bs = 200
jitter = 30

setting_id = 0
data_type = torch.float

parameters = dict()
parameters["resolution"] = 224
parameters["random_label"] = False
parameters["start_noise"] = True
parameters["detach_student"] = False
parameters["do_flip"] = True

parameters["store_best_images"] = True

criterion = nn.CrossEntropyLoss()


coefficients = dict()


# 通常ver
# coefficients["r_feature"] = 1.5
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.3
# coefficients["main_loss_multiplier"] = 17.5
# coefficients["feat_div"] = 2.0
# coefficients["proto_l2"] = 5.0
# coefficients["mmd"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v1"

# ver1 から feat_div を 0 に変更
# coefficients["r_feature"] = 1.5
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.3
# coefficients["main_loss_multiplier"] = 17.5
# coefficients["feat_div"] = 0.0
# coefficients["proto_l2"] = 5.0
# coefficients["mmd"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v2"

# ver2 から mmd を増加
# coefficients["r_feature"] = 1.5
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.3
# coefficients["main_loss_multiplier"] = 17.5
# coefficients["feat_div"] = 0.0
# coefficients["proto_l2"] = 5.0
# coefficients["mmd"] = 5.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v3"

# ver3 から proto_l2 を 0 に変更
# coefficients["r_feature"] = 1.5
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.3
# coefficients["main_loss_multiplier"] = 17.5
# coefficients["feat_div"] = 0.0
# coefficients["proto_l2"] = 0.0
# coefficients["mmd"] = 5.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v4"

# ver4 から proto_l2 を 増加
# coefficients["r_feature"] = 1.5
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.3
# coefficients["main_loss_multiplier"] = 17.5
# coefficients["feat_div"] = 0.0
# coefficients["proto_l2"] = 2.5
# coefficients["mmd"] = 5.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v5"

# ver6 から mmd を 増加
# coefficients["r_feature"] = 1.5
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.3
# coefficients["main_loss_multiplier"] = 17.5
# coefficients["feat_div"] = 0.0
# coefficients["proto_l2"] = 2.5
# coefficients["mmd"] = 10.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v6"

# ver6 から mmd を 増加
# coefficients["r_feature"] = 1.5
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.3
# coefficients["main_loss_multiplier"] = 17.5
# coefficients["feat_div"] = 0.0
# coefficients["proto_l2"] = 2.5
# coefficients["mmd"] = 15.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v7"

# ver7 から mmd を 増加
# coefficients["r_feature"] = 1.5
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.3
# coefficients["main_loss_multiplier"] = 17.5
# coefficients["feat_div"] = 0.0
# coefficients["proto_l2"] = 2.5
# coefficients["mmd"] = 20.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v8"

# ver8 から mmd を 増加
# coefficients["r_feature"] = 1.5
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.3
# coefficients["main_loss_multiplier"] = 17.5
# coefficients["feat_div"] = 0.0
# coefficients["proto_l2"] = 2.5
# coefficients["mmd"] = 25.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v9"

# ver9 から proto_l2 を 0 に変更
# coefficients["r_feature"] = 1.5
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.3
# coefficients["main_loss_multiplier"] = 17.5
# coefficients["feat_div"] = 0.0
# coefficients["proto_l2"] = 0.0
# coefficients["mmd"] = 25.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v10"


# ver10 から mmd を 増加
# coefficients["r_feature"] = 1.5
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.3
# coefficients["main_loss_multiplier"] = 17.5
# coefficients["feat_div"] = 0.0
# coefficients["proto_l2"] = 0.0
# coefficients["mmd"] = 35.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v11"

# ver11 から mmd を 増加
coefficients["r_feature"] = 1.5
coefficients["first_bn_multiplier"] = 10
coefficients["tv_l1"] = 0.0
coefficients["tv_l2"] = 0.001
coefficients["l2"] = 0.00001
coefficients["lr"] = 0.3
coefficients["main_loss_multiplier"] = 17.5
coefficients["feat_div"] = 0.0
coefficients["proto_l2"] = 0.0
coefficients["mmd"] = 50.0
coefficients["adi_scale"] = 0.0
coefficients["exp_descr"] = "debug_imnet100_v12"


network_output_function = lambda x: x


prefix = "runs/data_generation_di++++++_imnet/"+coefficients["exp_descr"]+"/"

for create_folder in [prefix, prefix+"/best_images/"]:
    if not os.path.exists(create_folder):
        os.makedirs(create_folder)

In [12]:
from utils.data_manager import DataManager

# === 実画像の特徴抽出（MMD 用のリアル特徴バンク） ===
from pathlib import Path
from torch.utils.data import DataLoader
import torch

# 1クラスあたり何サンプル集めるか（必要に応じて変更）
N_PER_CLASS = 300


# plot_feature_imnet100.ipynb と同じ形で DataManager を構築
data_manager = DataManager(
    dataset_name=args["dataset"],
    shuffle=args["shuffle"],
    seed=args["seed"][0],
    init_cls=args["init_cls"],
    increment=args["increment"],
)


# 対象とするクラス集合（手動で調整する）
num_classes = data_manager.get_total_classnum()
# target_classes = list(range(num_classes))
target_classes = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

print("total classes:", num_classes)
print("target classes:", target_classes)


# --------------------------------------------------
# 2. DataManager から実画像の Dataset を作成
#    - source="train": 訓練データ
#    - mode="test"   : ランダム変換なし（安定した特徴のため）
# --------------------------------------------------
real_dataset = data_manager.get_dataset(
    indices=target_classes,
    source="train",
    mode="test",
)

real_loader = DataLoader(
    real_dataset,
    batch_size=128,
    shuffle=True,    # クラスをランダムに拾うためシャッフル
    num_workers=4,   # 環境に合わせて調整
)


# --------------------------------------------------
# 3. モデルを評価モードにし，デバイスを決定
# --------------------------------------------------
net.eval()
# device = getattr(net, "_device", args["device"][0])
# device = learner._device
device='cuda'

# クラスごとに特徴ベクトルを貯める辞書
real_features = {c: [] for c in target_classes}


# --------------------------------------------------
# 4. DataLoader を回して特徴を抽出
#    - DummyDataset の __getitem__ は (idx, image, label) を返す
# --------------------------------------------------
with torch.no_grad():
    for idxs, images, labels in real_loader:
        images = images.to(device)
        labels = labels.to(device)

        # ネットワークから特徴を取得
        outputs = net(images)
        feats = outputs["features"]             # 形: (B, C, H, W) or (B, D)
        feats = feats.view(feats.size(0), -1)   # (B, D) に flatten

        # クラスごとに N_PER_CLASS 個まで集める
        for f, y in zip(feats.cpu(), labels.cpu().tolist()):
            if y in real_features and len(real_features[y]) < N_PER_CLASS:
                real_features[y].append(f)

        # すべてのクラスで目標数に達したらループ終了
        if all(len(real_features[c]) >= N_PER_CLASS for c in target_classes):
            break


# --------------------------------------------------
# 5. list -> tensor に変換して保存
# --------------------------------------------------
# 特徴の次元 D（少なくとも1クラスは埋まっている前提）
some_class = next(iter(target_classes))
if len(real_features[some_class]) > 0:
    feat_dim = real_features[some_class][0].numel()
else:
    feat_dim = 0

for c in target_classes:
    if len(real_features[c]) > 0:
        real_features[c] = torch.stack(real_features[c], dim=0)  # (N_c, D)
    else:
        real_features[c] = torch.empty(0, feat_dim)

# 保存ディレクトリとファイルパス
save_dir = Path("./mmd_real_features")
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "real_features_imnet100.pth"

torch.save(
    {
        "features": real_features,   # dict: class_id -> (N_c, D) tensor
        "n_per_class": N_PER_CLASS,
        "classes": target_classes,
    },
    save_path,
)

print("Saved real features for MMD to:", save_path)
for c in target_classes[:10]:
    print(f"class {c}: {real_features[c].shape[0]} samples")

total classes: 100
target classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Saved real features for MMD to: mmd_real_features/real_features_imnet100.pth
class 0: 300 samples
class 1: 300 samples
class 2: 300 samples
class 3: 300 samples
class 4: 300 samples
class 5: 300 samples
class 6: 300 samples
class 7: 300 samples
class 8: 300 samples
class 9: 300 samples


In [23]:
def get_image_prior_losses(inputs_jit):
    # COMPUTE total variation regularization loss
    diff1 = inputs_jit[:, :, :, :-1] - inputs_jit[:, :, :, 1:]
    diff2 = inputs_jit[:, :, :-1, :] - inputs_jit[:, :, 1:, :]
    diff3 = inputs_jit[:, :, 1:, :-1] - inputs_jit[:, :, :-1, 1:]
    diff4 = inputs_jit[:, :, :-1, :-1] - inputs_jit[:, :, 1:, 1:]

    loss_var_l2 = torch.norm(diff1) + torch.norm(diff2) + torch.norm(diff3) + torch.norm(diff4)
    loss_var_l1 = (diff1.abs() / 255.0).mean() + (diff2.abs() / 255.0).mean() + (
            diff3.abs() / 255.0).mean() + (diff4.abs() / 255.0).mean()
    loss_var_l1 = loss_var_l1 * 255.0
    return loss_var_l1, loss_var_l2


## Create hooks for feature statistics catching
loss_r_feature_layers = []
for module in net.modules():
    if isinstance(module, nn.BatchNorm2d):
        loss_r_feature_layers.append(DeepInversionFeatureHook(module))


best_cost = 1e4

# ラベルの用意
targets = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
# targets = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
# targets = [20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
# targets = [30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
# targets = [40, 41, 42, 43, 44, 45, 46, 47, 48, 49]

targets = torch.LongTensor(targets * (int(bs / len(targets)))).to('cuda')


# ============================================
# プロトタイプを用意（通常のDeepInversionと異なる箇所）
# ============================================
protos_dict = net._protos  # {label: proto_vec}
proto_labels = sorted(protos_dict.keys())
# print(protos_dict.keys())
# print(proto_labels)

# 各 proto を tensor にして並べる
protos_list = []
for c in proto_labels:
    v = protos_dict[c]                         # np.array or torch.Tensor
    v = torch.as_tensor(v, dtype=torch.float32)
    protos_list.append(v)

protos_tensor = torch.stack(protos_list, dim=0).to("cuda")  # (C, D)

# ラベル → 行index のマップを作っておく
label2row = {c: i for i, c in enumerate(proto_labels)}
C, D = protos_tensor.shape
print("num protos:", C, "feat dim:", D)

# ============================================
# プロトタイプを用意（通常のDeepInversionと異なる箇所）
# ============================================


# 実特徴バンクをロード（メモリに残っていればそれを使う）
if "real_features" not in globals():
    real_feat_path = Path("./mmd_real_features/real_features_imnet100.pth")
    print("Loading real features from:", real_feat_path)
    ckpt_real = torch.load(real_feat_path, map_location="cpu")
    real_features = ckpt_real["features"]  # dict: class_id -> (N_c, D) tensor
    print("Loaded classes:", ckpt_real["classes"])

# MMD 損失関数
MMD_REAL_PER_CLASS = 300
mmd_loss_fn = MMDLoss()
    

# 画像のサイズを取り出す
img_original = parameters["resolution"]

# seed値の固定
torch.manual_seed(777)
random.seed(777)

# 最適化する入力の用意
inputs = torch.randn((bs, 3, img_original, img_original), requires_grad=True, device='cuda', dtype=data_type)
pooling_function = nn.modules.pooling.AvgPool2d(kernel_size=2)

if setting_id==0:
    skipfirst = False
else:
    skipfirst = True

iteration = 0
for lr_it, lower_res in enumerate([2, 1]):
    if lr_it==0:
        iterations_per_layer = 2000
    else:
        iterations_per_layer = 1000 if not skipfirst else 2000
        if setting_id == 2:
            iterations_per_layer = 20000
    
    if lr_it==0 and skipfirst:
        continue

    lim_0, lim_1 = jitter // lower_res, jitter // lower_res

    if setting_id == 0:
        #multi resolution, 2k iterations with low resolution, 1k at normal, ResNet50v1.5 works the best, ResNet50 is ok
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.5, 0.9], eps = 1e-8)
        do_clip = True
    elif setting_id == 1:
        #2k normal resolultion, for ResNet50v1.5; Resnet50 works as well
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.5, 0.9], eps = 1e-8)
        do_clip = True
    elif setting_id == 2:
        #20k normal resolution the closes to the paper experiments for ResNet50
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.9, 0.999], eps = 1e-8)
        do_clip = False
    
    # 学習率スケジューラの設定
    lr_scheduler = lr_cosine_policy(coefficients["lr"], 100, iterations_per_layer)

    # 学習ループ
    for iteration_loc in range(iterations_per_layer):
        iteration += 1
        
        # 学習率の調整
        lr_scheduler(optimizer, iteration_loc, iteration_loc)

        # perform downsampling if needed
        if lower_res!=1:
            inputs_jit = pooling_function(inputs)
        else:
            inputs_jit = inputs

        # apply random jitter offsets
        off1 = random.randint(-lim_0, lim_0)
        off2 = random.randint(-lim_1, lim_1)
        inputs_jit = torch.roll(inputs_jit, shifts=(off1, off2), dims=(2, 3))

        # Flipping
        flip = random.random() > 0.5
        if flip and parameters["do_flip"]:
            inputs_jit = torch.flip(inputs_jit, dims=(3,))

        # 勾配のリセット
        optimizer.zero_grad()
        net.zero_grad()

        # forward処理
        outputs = net(inputs_jit)
        logits_all = outputs["logits"]
        logits = logits_all[:, ::4] 

        # 交差エントロピー損失の計算
        loss = criterion(logits, targets)

        # 特徴の多様性最大化損失（未完成）
        feature = outputs["features"]
        feat = feature.view(feature.size(0), -1)   # (bs, D)
        # print('feature.shape: ', feature.shape)
        # print("feat.shape: ", feat.shape)

        bs = feat.size(0)

        # ペアごとの差分ベクトル
        # print("feat.unsqueeze(1).shape: ", feat.unsqueeze(1).shape)
        # print("feat.unsqueeze(0).shape: ", feat.unsqueeze(0).shape)
        diff = feat.unsqueeze(1) - feat.unsqueeze(0)   # (bs, bs, D)
        dist2 = (diff ** 2).sum(dim=2)                 # (bs, bs), L2距離の二乗

        # 同一クラスかどうかのマスク
        same_label = (targets.unsqueeze(0) == targets.unsqueeze(1))  # (bs, bs) bool

        # 自分自身 (i == j) のペアは除外
        eye = torch.eye(bs, dtype=torch.bool, device=targets.device)
        same_label = same_label & (~eye)

        if same_label.any():
            same_dist2 = dist2[same_label]          # 同じクラス同士の距離だけ
            # 距離を「最大化」したいので，逆数を計算
            loss_div = 1.0 / same_dist2.mean()
        else:
            loss_div = torch.zeros(1, device=feat.device)
        

        # プロトタイプとの距離損失
        # protos_tensor: (C, D) をループ外で作っておいたものを使う
        # targets は net._protos の key と同じラベル空間と仮定
        row_idx = torch.tensor(
            [label2row[int(t.item())] for t in targets],
            device=feat.device,
            dtype=torch.long,
        )

        # 自分のクラスのプロトタイプベクトルを取り出す (bs, D)
        proto_pos = protos_tensor[row_idx]  # (bs, D)

        # ユークリッド距離（二乗）を計算
        # dist_i^2 = ||feat_i - proto_i||^2
        dist2 = torch.sum((feat - proto_pos) ** 2, dim=1)  # (bs,)

        # 平均距離（二乗）を最小化
        loss_eu = dist2.mean()
        loss_target_eu = loss_eu.item()

        
        # --------------------------
        # main loss 3: MMD (DI vs Real features)
        # --------------------------
        loss_mmd = compute_batch_mmd(feat, targets, real_features,
                                     max_real_per_class=MMD_REAL_PER_CLASS)


        # R_prior losses
        loss_var_l1, loss_var_l2 = get_image_prior_losses(inputs_jit)

        # R_feature loss
        rescale = [coefficients["first_bn_multiplier"]] + [1. for _ in range(len(loss_r_feature_layers)-1)]
        loss_r_feature = sum([mod.r_feature * rescale[idx] for (idx, mod) in enumerate(loss_r_feature_layers)])

        # l2 loss on images
        loss_l2 = torch.norm(inputs_jit.view(bs, -1), dim=1).mean()

        # combining losses
        loss_aux = coefficients["tv_l2"] * loss_var_l2 + \
                    coefficients["tv_l1"] * loss_var_l1 + \
                    coefficients["r_feature"] * loss_r_feature + \
                    coefficients["l2"] * loss_l2
                
        loss = coefficients["main_loss_multiplier"] * loss + coefficients["feat_div"] * loss_div + coefficients["proto_l2"]  * loss_eu + coefficients["mmd"] * loss_mmd + loss_aux

        if iteration % 10==0:
            print("------------iteration {}----------".format(iteration))
            print("total loss", loss.item())
            print("loss_r_feature", loss_r_feature.item())
            print("loss_div", loss_div.item())
            print("loss_proto", loss_target_eu)
            print("loss_mmd", loss_mmd.item())
            print("main criterion", criterion(logits, targets).item())

        loss.backward()
        optimizer.step()

        if do_clip:
            inputs.data = clip(inputs.data, use_fp16=False)


        if best_cost > loss.item() or iteration == 1:
            best_inputs = inputs.data.clone()
            best_cost = loss.item()

        if iteration % 100==0:
            vutils.save_image(inputs,
                                '{}/best_images/output_{:05d}_gpu.png'.format(prefix, iteration // 100,),
                                normalize=True, scale_each=True, nrow=int(10))


# 最適化した画像を保存
save_dir = os.path.join(prefix, "best_images")  # 画像を保存しているディレクトリと揃える例
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "deepinversion_inputs.pth")

to_save = {
    "images": best_inputs.detach().cpu(),   # 形: (bs, 3, H, W)
    "targets": targets.detach().cpu(),      # 対応するラベル
    "resolution": img_original,
    "classes": targets.unique().tolist(),   # どのクラスを生成したかのメモ（お好みで）
}
torch.save(to_save, save_path)
print("saved DeepInversion inputs to:", save_path)


# 最適化した画像を保存 ver2
all_exemplars = []
all_labels = []
with torch.no_grad():
    out = best_inputs.clone()

    # ImageNet 正規化を戻す
    mean = torch.tensor([0.485, 0.456, 0.406], device=feat.device).view(1, 3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225], device=feat.device).view(1, 3, 1, 1)
    out = out * std + mean
    out = torch.clamp(out, 0.0, 1.0)
    out = (out * 255.0).byte()
    out = out.permute(0, 2, 3, 1).cpu().numpy()  # (B, H, W, 3)

    all_exemplars.append(out)
    all_labels.append(targets.detach().cpu().numpy())

    save_data = np.concatenate(all_exemplars, axis=0)
    save_labels = np.concatenate(all_labels, axis=0)

    imgs_tensor = torch.from_numpy(save_data).permute(0, 3, 1, 2).contiguous()  # uint8
    labels_tensor = torch.from_numpy(save_labels.astype(np.int64))

    save_path = os.path.join(save_dir, "deepinversion_inputs_ver2.pth")

    save_obj = {
        "images": imgs_tensor,   # shape: (N, 3, H, W), dtype: uint8
        "labels": labels_tensor, # shape: (N,)
        "task": None,
    }
    torch.save(save_obj, save_path)

num protos: 50 feat dim: 512
------------iteration 10----------
total loss 697.3217163085938
loss_r_feature 393.8476867675781
loss_div 0.5711084008216858
loss_proto 12.162614822387695
loss_mmd 0.680016040802002
main criterion 3.7243728637695312
------------iteration 20----------
total loss 584.7247924804688
loss_r_feature 341.7925109863281
loss_div 0.05102948844432831
loss_proto 14.939852714538574
loss_mmd 0.23193395137786865
main criterion 3.0376760959625244
------------iteration 30----------
total loss 475.5638732910156
loss_r_feature 284.666748046875
loss_div 0.03228018060326576
loss_proto 19.92692756652832
loss_mmd 0.1751953810453415
main criterion 1.861363410949707
------------iteration 40----------
total loss 384.1267395019531
loss_r_feature 228.70501708984375
loss_div 0.027707906439900398
loss_proto 23.988204956054688
loss_mmd 0.1878989338874817
main criterion 1.4037809371948242
------------iteration 50----------
total loss 317.9373779296875
loss_r_feature 186.72515869140625
los

## 交差エントロピー損失+MMD+プロトタイプ損失
プロトタイプと擬似画像の平均特徴間で距離を最小化する損失

In [13]:
iterations = 4000
start_noise = True
# args.detach_student = False

resolution = 224
bs = 200
jitter = 30

setting_id = 0
data_type = torch.float

parameters = dict()
parameters["resolution"] = 224
parameters["random_label"] = False
parameters["start_noise"] = True
parameters["detach_student"] = False
parameters["do_flip"] = True

parameters["store_best_images"] = True

criterion = nn.CrossEntropyLoss()


coefficients = dict()


# 通常ver
# coefficients["r_feature"] = 1.5
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.3
# coefficients["main_loss_multiplier"] = 17.5
# coefficients["feat_div"] = 0.0
# coefficients["proto"] = 1.0
# coefficients["mmd"] = 50.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v1"

# ver1 から proto を増加 
# coefficients["r_feature"] = 1.5
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.3
# coefficients["main_loss_multiplier"] = 17.5
# coefficients["feat_div"] = 0.0
# coefficients["proto"] = 1.5
# coefficients["mmd"] = 50.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v2"

# ver2 から feat_div を増加 
# coefficients["r_feature"] = 1.5
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.3
# coefficients["main_loss_multiplier"] = 17.5
# coefficients["feat_div"] = 1.0
# coefficients["proto"] = 1.5
# coefficients["mmd"] = 50.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v3"

# ver3 から main_loss_multiplier を増加 
# coefficients["r_feature"] = 1.5
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.3
# coefficients["main_loss_multiplier"] = 20.0
# coefficients["feat_div"] = 1.0
# coefficients["proto"] = 1.5
# coefficients["mmd"] = 50.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v4"

# ver4 から feat_div　を変更 
# coefficients["r_feature"] = 1.5
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 20.0
# coefficients["feat_div"] = 1.5
# coefficients["proto"] = 1.5
# coefficients["mmd"] = 50.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v5"

# ver4 から mmd　を変更 
coefficients["r_feature"] = 1.5
coefficients["first_bn_multiplier"] = 10
coefficients["tv_l1"] = 0.0
coefficients["tv_l2"] = 0.001
coefficients["l2"] = 0.00001
coefficients["lr"] = 0.25
coefficients["main_loss_multiplier"] = 20.0
coefficients["feat_div"] = 1.5
coefficients["proto"] = 1.5
coefficients["mmd"] = 60.0
coefficients["adi_scale"] = 0.0
coefficients["exp_descr"] = "debug_imnet100_v6"


network_output_function = lambda x: x


prefix = "runs/data_generation_di-mmd_imnet/"+coefficients["exp_descr"]+"/"

for create_folder in [prefix, prefix+"/best_images/"]:
    if not os.path.exists(create_folder):
        os.makedirs(create_folder)

In [14]:
def get_image_prior_losses(inputs_jit):
    # COMPUTE total variation regularization loss
    diff1 = inputs_jit[:, :, :, :-1] - inputs_jit[:, :, :, 1:]
    diff2 = inputs_jit[:, :, :-1, :] - inputs_jit[:, :, 1:, :]
    diff3 = inputs_jit[:, :, 1:, :-1] - inputs_jit[:, :, :-1, 1:]
    diff4 = inputs_jit[:, :, :-1, :-1] - inputs_jit[:, :, 1:, 1:]

    loss_var_l2 = torch.norm(diff1) + torch.norm(diff2) + torch.norm(diff3) + torch.norm(diff4)
    loss_var_l1 = (diff1.abs() / 255.0).mean() + (diff2.abs() / 255.0).mean() + (
            diff3.abs() / 255.0).mean() + (diff4.abs() / 255.0).mean()
    loss_var_l1 = loss_var_l1 * 255.0
    return loss_var_l1, loss_var_l2


## Create hooks for feature statistics catching
loss_r_feature_layers = []
for module in net.modules():
    if isinstance(module, nn.BatchNorm2d):
        loss_r_feature_layers.append(DeepInversionFeatureHook(module))


best_cost = 1e4

# ラベルの用意
targets = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
# targets = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
# targets = [20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
# targets = [30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
# targets = [40, 41, 42, 43, 44, 45, 46, 47, 48, 49]

targets = torch.LongTensor(targets * (int(bs / len(targets)))).to('cuda')


# ============================================
# プロトタイプを用意（通常のDeepInversionと異なる箇所）
# ============================================
protos_dict = net._protos  # {label: proto_vec}
proto_labels = sorted(protos_dict.keys())
# print(protos_dict.keys())
# print(proto_labels)

# 各 proto を tensor にして並べる
protos_list = []
for c in proto_labels:
    v = protos_dict[c]                         # np.array or torch.Tensor
    v = torch.as_tensor(v, dtype=torch.float32)
    protos_list.append(v)

protos_tensor = torch.stack(protos_list, dim=0).to("cuda")  # (C, D)

# ラベル → 行index のマップを作っておく
label2row = {c: i for i, c in enumerate(proto_labels)}
C, D = protos_tensor.shape
print("num protos:", C, "feat dim:", D)

# ============================================
# プロトタイプを用意（通常のDeepInversionと異なる箇所）
# ============================================


# 実特徴バンクをロード（メモリに残っていればそれを使う）
if "real_features" not in globals():
    real_feat_path = Path("./mmd_real_features/real_features_imnet100.pth")
    print("Loading real features from:", real_feat_path)
    ckpt_real = torch.load(real_feat_path, map_location="cpu")
    real_features = ckpt_real["features"]  # dict: class_id -> (N_c, D) tensor
    print("Loaded classes:", ckpt_real["classes"])

# MMD 損失関数
MMD_REAL_PER_CLASS = 300
mmd_loss_fn = MMDLoss()
    

# 画像のサイズを取り出す
img_original = parameters["resolution"]

# seed値の固定
torch.manual_seed(777)
random.seed(777)

# 最適化する入力の用意
inputs = torch.randn((bs, 3, img_original, img_original), requires_grad=True, device='cuda', dtype=data_type)
pooling_function = nn.modules.pooling.AvgPool2d(kernel_size=2)

if setting_id==0:
    skipfirst = False
else:
    skipfirst = True

iteration = 0
for lr_it, lower_res in enumerate([2, 1]):
    if lr_it==0:
        iterations_per_layer = 2000
    else:
        iterations_per_layer = 1000 if not skipfirst else 2000
        if setting_id == 2:
            iterations_per_layer = 20000
    
    if lr_it==0 and skipfirst:
        continue

    lim_0, lim_1 = jitter // lower_res, jitter // lower_res

    if setting_id == 0:
        #multi resolution, 2k iterations with low resolution, 1k at normal, ResNet50v1.5 works the best, ResNet50 is ok
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.5, 0.9], eps = 1e-8)
        do_clip = True
    elif setting_id == 1:
        #2k normal resolultion, for ResNet50v1.5; Resnet50 works as well
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.5, 0.9], eps = 1e-8)
        do_clip = True
    elif setting_id == 2:
        #20k normal resolution the closes to the paper experiments for ResNet50
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.9, 0.999], eps = 1e-8)
        do_clip = False
    
    # 学習率スケジューラの設定
    lr_scheduler = lr_cosine_policy(coefficients["lr"], 100, iterations_per_layer)

    # 学習ループ
    for iteration_loc in range(iterations_per_layer):
        iteration += 1
        
        # 学習率の調整
        lr_scheduler(optimizer, iteration_loc, iteration_loc)

        # perform downsampling if needed
        if lower_res!=1:
            inputs_jit = pooling_function(inputs)
        else:
            inputs_jit = inputs

        # apply random jitter offsets
        off1 = random.randint(-lim_0, lim_0)
        off2 = random.randint(-lim_1, lim_1)
        inputs_jit = torch.roll(inputs_jit, shifts=(off1, off2), dims=(2, 3))

        # Flipping
        flip = random.random() > 0.5
        if flip and parameters["do_flip"]:
            inputs_jit = torch.flip(inputs_jit, dims=(3,))

        # 勾配のリセット
        optimizer.zero_grad()
        net.zero_grad()

        # forward処理
        outputs = net(inputs_jit)
        logits_all = outputs["logits"]
        logits = logits_all[:, ::4] 

        # 交差エントロピー損失の計算
        loss = criterion(logits, targets)

        # 特徴の多様性最大化損失（未完成）
        feature = outputs["features"]
        feat = feature.view(feature.size(0), -1)   # (bs, D)
        # print('feature.shape: ', feature.shape)
        # print("feat.shape: ", feat.shape)

        bs = feat.size(0)

        # ペアごとの差分ベクトル
        # print("feat.unsqueeze(1).shape: ", feat.unsqueeze(1).shape)
        # print("feat.unsqueeze(0).shape: ", feat.unsqueeze(0).shape)
        diff = feat.unsqueeze(1) - feat.unsqueeze(0)   # (bs, bs, D)
        dist2 = (diff ** 2).sum(dim=2)                 # (bs, bs), L2距離の二乗

        # 同一クラスかどうかのマスク
        same_label = (targets.unsqueeze(0) == targets.unsqueeze(1))  # (bs, bs) bool

        # 自分自身 (i == j) のペアは除外
        eye = torch.eye(bs, dtype=torch.bool, device=targets.device)
        same_label = same_label & (~eye)

        if same_label.any():
            same_dist2 = dist2[same_label]          # 同じクラス同士の距離だけ
            # 距離を「最大化」したいので，逆数を計算
            loss_div = 1.0 / same_dist2.mean()
        else:
            loss_div = torch.zeros(1, device=feat.device)
        

        # --------------------------
        # main loss 2: プロトタイプと平均特徴の損失（未完成）
        # --------------------------
        loss_proto_list = []

        uniq_classes = targets.unique().tolist()
        for c in uniq_classes:
            c_int = int(c)
            # このクラスの DI 特徴を集める
            mask_c = (targets == c)
            feat_c = feat[mask_c]                 # (n_c, D)
            if feat_c.shape[0] == 0:
                continue

            # クラス c の DI 特徴の平均 μ_c
            feat_mean_c = feat_c.mean(dim=0)      # (D,)

            # 対応するプロトタイプ p_c を取得
            if c_int not in label2row:
                continue
            proto_c = protos_tensor[label2row[c_int]]  # (D,)

            # μ_c と p_c の L2 距離^2
            loss_c = ((feat_mean_c - proto_c) ** 2).sum()
            loss_proto_list.append(loss_c)

        if len(loss_proto_list) == 0:
            loss_proto = torch.tensor(0.0, device=feat.device)
        else:
            # クラス平均
            loss_proto = torch.stack(loss_proto_list).mean()

        
        # --------------------------
        # main loss 3: MMD (DI vs Real features)
        # --------------------------
        loss_mmd = compute_batch_mmd(feat, targets, real_features,
                                     max_real_per_class=MMD_REAL_PER_CLASS)


        # R_prior losses
        loss_var_l1, loss_var_l2 = get_image_prior_losses(inputs_jit)

        # R_feature loss
        rescale = [coefficients["first_bn_multiplier"]] + [1. for _ in range(len(loss_r_feature_layers)-1)]
        loss_r_feature = sum([mod.r_feature * rescale[idx] for (idx, mod) in enumerate(loss_r_feature_layers)])

        # l2 loss on images
        loss_l2 = torch.norm(inputs_jit.view(bs, -1), dim=1).mean()

        # combining losses
        loss_aux = coefficients["tv_l2"] * loss_var_l2 + \
                    coefficients["tv_l1"] * loss_var_l1 + \
                    coefficients["r_feature"] * loss_r_feature + \
                    coefficients["l2"] * loss_l2
                
        loss = coefficients["main_loss_multiplier"] * loss + coefficients["feat_div"] * loss_div + coefficients["proto"]  * loss_proto + coefficients["mmd"] * loss_mmd + loss_aux

        if iteration % 10==0:
            print("------------iteration {}----------".format(iteration))
            print("total loss", loss.item())
            print("loss_r_feature", loss_r_feature.item())
            print("loss_div", loss_div.item())
            print("loss_proto", loss_proto.item())
            print("loss_mmd", loss_mmd.item())
            print("main criterion", criterion(logits, targets).item())

        loss.backward()
        optimizer.step()

        if do_clip:
            inputs.data = clip(inputs.data, use_fp16=False)


        if best_cost > loss.item() or iteration == 1:
            best_inputs = inputs.data.clone()
            best_cost = loss.item()

        if iteration % 100==0:
            vutils.save_image(inputs,
                                '{}/best_images/output_{:05d}_gpu.png'.format(prefix, iteration // 100,),
                                normalize=True, scale_each=True, nrow=int(10))


# 最適化した画像を保存
save_dir = os.path.join(prefix, "best_images")  # 画像を保存しているディレクトリと揃える例
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "deepinversion_inputs.pth")

to_save = {
    "images": best_inputs.detach().cpu(),   # 形: (bs, 3, H, W)
    "targets": targets.detach().cpu(),      # 対応するラベル
    "resolution": img_original,
    "classes": targets.unique().tolist(),   # どのクラスを生成したかのメモ（お好みで）
}
torch.save(to_save, save_path)
print("saved DeepInversion inputs to:", save_path)


# 最適化した画像を保存 ver2
all_exemplars = []
all_labels = []
with torch.no_grad():
    out = best_inputs.clone()

    # ImageNet 正規化を戻す
    mean = torch.tensor([0.485, 0.456, 0.406], device=feat.device).view(1, 3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225], device=feat.device).view(1, 3, 1, 1)
    out = out * std + mean
    out = torch.clamp(out, 0.0, 1.0)
    out = (out * 255.0).byte()
    out = out.permute(0, 2, 3, 1).cpu().numpy()  # (B, H, W, 3)

    all_exemplars.append(out)
    all_labels.append(targets.detach().cpu().numpy())

    save_data = np.concatenate(all_exemplars, axis=0)
    save_labels = np.concatenate(all_labels, axis=0)

    imgs_tensor = torch.from_numpy(save_data).permute(0, 3, 1, 2).contiguous()  # uint8
    labels_tensor = torch.from_numpy(save_labels.astype(np.int64))

    save_path = os.path.join(save_dir, "deepinversion_inputs_ver2.pth")

    save_obj = {
        "images": imgs_tensor,   # shape: (N, 3, H, W), dtype: uint8
        "labels": labels_tensor, # shape: (N,)
        "task": None,
    }
    torch.save(save_obj, save_path)

num protos: 50 feat dim: 512
------------iteration 10----------
total loss 747.2185668945312
loss_r_feature 403.5090637207031
loss_div 0.40962523221969604
loss_proto 12.015052795410156
loss_mmd 0.6765891909599304
main criterion 3.7656450271606445
------------iteration 20----------
total loss 638.6956176757812
loss_r_feature 359.69232177734375
loss_div 0.06304536759853363
loss_proto 6.068042755126953
loss_mmd 0.2631545662879944
main criterion 3.342373847961426
------------iteration 30----------
total loss 533.5424194335938
loss_r_feature 311.6338806152344
loss_div 0.035938750952482224
loss_proto 3.5284829139709473
loss_mmd 0.14872346818447113
main criterion 2.2291696071624756
------------iteration 40----------
total loss 444.36065673828125
loss_r_feature 260.35003662109375
loss_div 0.034442830830812454
loss_proto 3.3409950733184814
loss_mmd 0.13787934184074402
main criterion 1.6647017002105713
------------iteration 50----------
total loss 365.8914489746094
loss_r_feature 211.09700012207